## Пример комбинирования Word2Vec и LDA

Использование Word2Vec в контексте LDA  
Word2Vec может быть косвенно связан с LDA, если мы используем эмбеддинги слов для:
- предобработки текстов перед обучением LDA (например, отфильтровывать нерелевантные слова или кластеризовать документы на основе похожести).
- улучшения визуализации тем, создаваемых LDA, через снижение размерности эмбеддингов.
- Обогащения признаков: в LDA можно использовать эмбеддинги слов как дополнительные признаки.
_____________________________________
__Word2Vec:__ Обучает модель для создания эмбеддингов слов. Фильтрация слов: Используем Word2Vec для выбора только тех слов, которые семантически связаны с важной темой.  
__LDA:__ Обучаем LDA на отфильтрованных текстах для выделения тем.
Таким образом, Word2Vec и LDA можно использовать вместе, если требуется обогащение данных или улучшение качества тематического моделирования.

## Импорт и подготовка данных -> <font color='yellow'>tokenized_texts</font>

In [63]:
from gensim.models import Word2Vec, LdaModel
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from nltk.tokenize import word_tokenize
import nltk
import pandas as pd

In [64]:
# nltk.download('punkt_tab')  # Загрузка токенизатора
nltk.download('punkt')  # Загрузка токенизатора

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shaps\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [65]:
# Пример корпуса
texts = [
    "Natural language processing with Word2Vec and LDA.",
    "Word embeddings are useful for semantic analysis.",
    "Topic modeling with LDA is powerful.",
    "Gensim is a great library for NLP tasks."
]

In [66]:
if not all(isinstance(doc, str) for doc in texts):
    raise ValueError("All elements in 'texts' must be strings")

In [ ]:
example = ["hello", "world", 123]  # В списке есть не строка (число 123)
if not all(isinstance(doc, str) for doc in example):
    raise ValueError("All elements in 'texts' must be strings")


In [67]:
# Токенизация
tokenized_texts = [word_tokenize(doc.lower()) for doc in texts]

In [68]:
tokenized_texts

[['natural', 'language', 'processing', 'with', 'word2vec', 'and', 'lda', '.'],
 ['word', 'embeddings', 'are', 'useful', 'for', 'semantic', 'analysis', '.'],
 ['topic', 'modeling', 'with', 'lda', 'is', 'powerful', '.'],
 ['gensim', 'is', 'a', 'great', 'library', 'for', 'nlp', 'tasks', '.']]

## __________________ word2vec -> <font color='lightgreen'>filtered_texts</font> _________________________

In [19]:
# Обучение Word2Vec
word2vec_model = Word2Vec(sentences=tokenized_texts, vector_size=100, window=5, min_count=1, workers=4)

### Параметры:
- __sentences=tokenized_texts__: Это список предложений, где каждое предложение представлено как список токенов (слов). На основе этих данных Word2Vec строит векторные представления слов.
- __vector_size=100__: Определяет размерность создаваемых векторов слов (то есть количество чисел в каждом векторе).
Чем больше размерность, тем больше информации может быть закодировано в векторе, но это также увеличивает вычислительную нагрузку и риск переобучения. Типичные значения: 50–300.
- __window=5__: Устанавливает размер "окна контекста".Окно контекста определяет, сколько слов по обе стороны от целевого слова будет учитываться для определения его контекста. Например, при window=5 слово будет связано с ближайшими 5 словами слева и 5 словами справа.
- __min_count=1__: Минимальная частота появления слова в тексте для включения его в словарь модели.
Слова, которые встречаются реже указанного значения, игнорируются. Например, при min_count=1 включаются все слова, встречающиеся хотя бы 1 раз.
- __workers=4__: Количество потоков (ядер процессора), которые используются для обучения. Чем больше ядер, тем быстрее проходит обучение, если ваш процессор поддерживает многопоточность.


#### Что мы получили

In [70]:
# К-во слов в словаре
vocab_len = len(word2vec_model.wv)
print(f"К-во слов в словаре: {vocab_len}") 
# Какой индекс у конкретного слова
can_idx = word2vec_model.wv.key_to_index["processing"]
print(f"Какой индекс у слова 'processing': {can_idx}") 
# Посчитать, сколько раз конкретное слова встретилось в документе(документах)
tutorials_cnt = word2vec_model.wv.get_vecattr("processing", "count")
print(f"Сколько раз слово 'processing' встретилось в документах: {tutorials_cnt}") 
# Косинусное сходство
similarity_1 = word2vec_model.wv.similarity("processing", "language")
similarity_2 = word2vec_model.wv.similarity("processing", "great")
print(f"Косинусное сходство 'processing', 'language': {similarity_1}")
print(f"Косинусное сходство 'processing', 'great': {similarity_2}")
# Вектор слова
# print(f"Вектор слова processing: {word2vec_model.wv["processing"]}")

К-во слов в словаре: 25
Какой индекс у слова 'processing': 7
Сколько раз слово 'processing' встретилось в документах: 1
Косинусное сходство 'processing', 'language': 0.005586662795394659
Косинусное сходство 'processing', 'great': 0.015761088579893112


In [72]:
# Посмотреть матрицу эмбеддинга
embedding_matrix = word2vec_model.wv.vectors
print(f"Форма матрицы эмбеддинга - {embedding_matrix.shape}")
print(f"Тип матрицы эмбеддинга - {type(embedding_matrix)}")
# print(f"Один документ - {embedding_matrix[:1]}")

Форма матрицы эмбеддинга - (25, 100)
Тип матрицы эмбеддинга - <class 'numpy.ndarray'>


In [41]:
# Какие слова соответствуют строкам матрицы:
words = word2vec_model.wv.index_to_key # Это словарь
print(words)  # Список всех слов в словаре
print(type(words))  # Список всех слов в словаре
print(len(words))  # Список всех слов в словаре

['.', 'for', 'with', 'is', 'lda', 'useful', 'language', 'processing', 'word2vec', 'and', 'word', 'embeddings', 'are', 'tasks', 'nlp', 'semantic', 'analysis', 'topic', 'modeling', 'powerful', 'gensim', 'a', 'great', 'library', 'natural']
<class 'list'>
25


In [ ]:
# Вычисление матрицы схожести
similarity_matrix = []
for word1 in words:
    row = [word2vec_model.wv.similarity(word1, word2) for word2 in words]
    similarity_matrix.append(row)
# Создание таблицы (DataFrame)
similarity_df = pd.DataFrame(similarity_matrix, index=words, columns=words)
similarity_df

In [88]:
# Отфильтруем слова с низкой семантической похожестью
# Например, оставим только слова, которые имеют похожесть с "embeddings"
# Если topn=10, вернутся 10 наиболее похожих слов (если такие существуют).
# Если topn больше количества слов в модели, метод вернёт столько слов, сколько возможно.
similar_words = [word for word, similarity in word2vec_model.wv.most_similar('embeddings', topn=25) if similarity > 0]
len(similar_words)

15

In [ ]:
similar_words

In [89]:
# Фильтрация исходных текстов
filtered_texts = [[word for word in doc if word in similar_words] for doc in tokenized_texts]
filtered_texts

[['natural', 'processing', 'with', 'word2vec', '.'],
 ['word', 'useful', 'for', 'semantic', '.'],
 ['topic', 'modeling', 'with', 'is', 'powerful', '.'],
 ['gensim', 'is', 'a', 'for', '.']]

## __________________ LDA -> <font color='lightgreen'>tokenized_texts,</font><font color='yellow'> filtered_texts</font>_________________________

### Создание словаря для LDA

In [102]:
dictionary_1 = Dictionary(tokenized_texts)
corpus_1 = [dictionary_1.doc2bow(text) for text in tokenized_texts]

In [103]:
dictionary_2 = Dictionary(filtered_texts)
corpus_2 = [dictionary_2.doc2bow(text) for text in filtered_texts]

In [99]:
print(dictionary_2.token2id)

{'.': 0, 'natural': 1, 'processing': 2, 'with': 3, 'word2vec': 4, 'for': 5, 'semantic': 6, 'useful': 7, 'word': 8, 'is': 9, 'modeling': 10, 'powerful': 11, 'topic': 12, 'a': 13, 'gensim': 14}


In [104]:
print(corpus_1)
print('____________________')
print(corpus_2)

[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1)], [(0, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1)], [(0, 1), (3, 1), (6, 1), (15, 1), (16, 1), (17, 1), (18, 1)], [(0, 1), (11, 1), (15, 1), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1)]]
____________________
[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1)], [(0, 1), (5, 1), (6, 1), (7, 1), (8, 1)], [(0, 1), (3, 1), (9, 1), (10, 1), (11, 1), (12, 1)], [(0, 1), (5, 1), (9, 1), (13, 1), (14, 1)]]


### Обучение LDA

In [107]:
lda_model_1 = LdaModel(corpus=corpus_1, id2word=dictionary_1, num_topics=2, passes=10)

In [106]:
lda_model_2 = LdaModel(corpus=corpus_2, id2word=dictionary_2, num_topics=2, passes=10)

### Получение тем

In [108]:
topics = lda_model_1.print_topics()
for topic in topics:
    print(topic)

(0, '0.085*"." + 0.085*"for" + 0.051*"is" + 0.051*"a" + 0.051*"tasks" + 0.051*"great" + 0.051*"nlp" + 0.051*"library" + 0.051*"gensim" + 0.051*"are"')
(1, '0.091*"." + 0.091*"with" + 0.091*"lda" + 0.054*"processing" + 0.054*"modeling" + 0.054*"powerful" + 0.054*"and" + 0.054*"natural" + 0.054*"language" + 0.054*"word2vec"')


In [109]:
topics = lda_model_2.print_topics()
for topic in topics:
    print(topic)

(0, '0.143*"." + 0.086*"for" + 0.086*"with" + 0.085*"useful" + 0.085*"semantic" + 0.085*"word" + 0.085*"natural" + 0.085*"word2vec" + 0.085*"processing" + 0.029*"a"')
(1, '0.135*"is" + 0.135*"." + 0.081*"with" + 0.081*"modeling" + 0.081*"powerful" + 0.081*"topic" + 0.081*"for" + 0.081*"gensim" + 0.081*"a" + 0.027*"processing"')


### Оценка согласованности модели

In [110]:
coherence_model = CoherenceModel(model=lda_model_1, texts=tokenized_texts, dictionary=dictionary_1, coherence='c_v')
coherence = coherence_model.get_coherence()
print(f"Coherence score: {coherence}")

Coherence score: 0.33250373230059616


In [111]:
coherence_model = CoherenceModel(model=lda_model_2, texts=filtered_texts, dictionary=dictionary_2, coherence='c_v')
coherence = coherence_model.get_coherence()
print(f"Coherence score: {coherence}")

Coherence score: 0.3725101306326768
